# Panel Construction: Congressional Trading Database

**Proyecto:** Detección de Insider Trading en el Congreso de EE.UU.  
**Nivel:** (Ticker, Month)  
**Output:** `panel_final_stock_month.parquet`

---

## Pipeline

```
congress_trades_with_committees.parquet (nivel trade, ~87K obs)
    │
    ├── [1] Limpieza y preparación
    ├── [2] Features de congreso a nivel trade
    ├── [3] Colapso a nivel (ticker, month) - Variables de congreso
    ├── [4] Colapso a nivel (ticker, month) - Variables de mercado (fin de mes)
    ├── [5] Merge y creación de variables derivadas
    ├── [6] Variables objetivo (ret_future_1m, CAR)
    ├── [7] Winsorizing y validación
    └── [8] Export
    │
    ▼
panel_final_stock_month.parquet (~32K obs)
```

---

In [2]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("PANEL CONSTRUCTION: CONGRESSIONAL TRADING DATABASE")
print("="*70)

PANEL CONSTRUCTION: CONGRESSIONAL TRADING DATABASE


In [3]:
# =============================================================================
# CONFIGURACIÓN
# =============================================================================

# Paths
INPUT_PATH = 'data/outputs/congress_trades_with_committees.parquet'
OUTPUT_DIR = 'data/prediction_bases'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Período de análisis
START_DATE = '2012-01-01'
END_DATE = '2024-12-31'

# Winsorizing
WINSOR_PCTL = 0.01  # 1% y 99%

# Comités con información privilegiada potencial
INFO_COMMITTEES = [
    'Armed Services', 'Financial Services', 'Energy and Commerce',
    'Intelligence', 'Select Committee on Intelligence',
    'Ways and Means', 'Appropriations', 'Health Education Labor and Pensions',
    'Banking, Housing and Urban Affairs', 'Finance',
    'Judiciary', 'Commerce, Science and Transportation'
]

print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Período: {START_DATE} a {END_DATE}")

Input: data/outputs/congress_trades_with_committees.parquet
Output: data/prediction_bases
Período: 2012-01-01 a 2024-12-31


---
## 1. Load & Clean Data

In [4]:
print("\n[1] CARGANDO DATOS...")

df = pd.read_parquet(INPUT_PATH)
n_initial = len(df)

print(f"    Trades cargados: {n_initial:,}")
print(f"    Columnas: {len(df.columns)}")


[1] CARGANDO DATOS...
    Trades cargados: 99,609
    Columnas: 97


In [5]:
# Mostrar columnas disponibles
print("\n    Columnas disponibles:")
for i, col in enumerate(df.columns):
    print(f"      {i+1:2}. {col}")


    Columnas disponibles:
       1. Ticker
       2. TickerType
       3. Company
       4. Traded
       5. Transaction
       6. Trade_Size_USD
       7. Status
       8. Subholding
       9. Description
      10. Name
      11. BioGuideID
      12. Filed
      13. Party
      14. District
      15. Chamber
      16. Comments
      17. Quiver_Upload_Time
      18. excess_return
      19. State
      20. last_modified
      21. Ticker_Clean
      22. is_equity
      23. trade_id
      24. return_t
      25. abs_return_t
      26. return_overnight
      27. return_intraday
      28. momentum_5d
      29. momentum_20d
      30. momentum_60d
      31. momentum_252d
      32. realized_vol_30d
      33. parkinson_vol_30d
      34. realized_vol_60d
      35. vol_of_vol_60d
      36. realized_vol_252d
      37. volume_t
      38. dollar_volume_t
      39. volume_ratio_30d
      40. abnormal_volume_30d
      41. amihud_illiq_20d
      42. roll_spread_30d
      43. hl_spread_20d
      44. zer

In [6]:
# --- Identificar columnas clave ---
print("\n    Identificando columnas...")

# Fecha de trade
if 'Traded' in df.columns:
    df['trade_date'] = pd.to_datetime(df['Traded'])
elif 'trade_date' in df.columns:
    df['trade_date'] = pd.to_datetime(df['trade_date'])
else:
    raise ValueError("No se encontró columna de fecha de trade")

# Fecha de filing
if 'Filed' in df.columns:
    df['filed_date'] = pd.to_datetime(df['Filed'])
else:
    df['filed_date'] = df['trade_date']

# Ticker
if 'Ticker_Clean' in df.columns:
    TICKER_COL = 'Ticker_Clean'
elif 'Ticker' in df.columns:
    df['Ticker_Clean'] = df['Ticker'].str.upper().str.strip()
    TICKER_COL = 'Ticker_Clean'
else:
    raise ValueError("No se encontró columna de ticker")

# Nombre del político
name_candidates = ['full_name', 'Name', 'name', 'politician']
NAME_COL = None
for col in name_candidates:
    if col in df.columns:
        NAME_COL = col
        break
if NAME_COL is None:
    NAME_COL = [c for c in df.columns if 'name' in c.lower()][0]

print(f"      Ticker: {TICKER_COL}")
print(f"      Nombre: {NAME_COL}")
print(f"      Fecha trade: trade_date")
print(f"      Fecha filing: filed_date")


    Identificando columnas...
      Ticker: Ticker_Clean
      Nombre: full_name
      Fecha trade: trade_date
      Fecha filing: filed_date


In [7]:
# --- Filtrar período ---
df = df[(df['trade_date'] >= START_DATE) & (df['trade_date'] <= END_DATE)].copy()

# Crear trade_month
df['trade_month'] = df['trade_date'].dt.to_period('M')
df['trade_year'] = df['trade_date'].dt.year

print(f"\n    Trades en período: {len(df):,} (de {n_initial:,})")
print(f"    Período: {df['trade_date'].min().date()} a {df['trade_date'].max().date()}")
print(f"    Acciones únicas: {df[TICKER_COL].nunique():,}")
print(f"    Políticos únicos: {df[NAME_COL].nunique():,}")


    Trades en período: 86,624 (de 99,609)
    Período: 2012-06-06 a 2024-12-31
    Acciones únicas: 4,440
    Políticos únicos: 246


---
## 2. Trade-Level Features (Congress)

In [8]:
print("\n[2] CREANDO FEATURES A NIVEL TRADE...")

# === 2.1 DIRECCIÓN ===
if 'Transaction' in df.columns:
    df['is_buy'] = df['Transaction'].str.lower().str.contains('purchase|buy', na=False).astype(int)
    df['is_sell'] = df['Transaction'].str.lower().str.contains('sale|sell', na=False).astype(int)
else:
    df['is_buy'] = 0
    df['is_sell'] = 0

df['trade_direction'] = df['is_buy'] - df['is_sell']

print(f"    Compras: {df['is_buy'].sum():,} ({df['is_buy'].mean()*100:.1f}%)")
print(f"    Ventas: {df['is_sell'].sum():,} ({df['is_sell'].mean()*100:.1f}%)")


[2] CREANDO FEATURES A NIVEL TRADE...
    Compras: 43,826 (50.6%)
    Ventas: 42,388 (48.9%)


In [9]:
# === 2.2 MONTO ===
if 'Trade_Size_USD' in df.columns:
    size_map = {
        '$1,001 - $15,000': 8000,
        '$15,001 - $50,000': 32500,
        '$50,001 - $100,000': 75000,
        '$100,001 - $250,000': 175000,
        '$250,001 - $500,000': 375000,
        '$500,001 - $1,000,000': 750000,
        '$1,000,001 - $5,000,000': 3000000,
        'Over $5,000,000': 7500000,
    }
    df['amount_proxy'] = df['Trade_Size_USD'].map(size_map).fillna(8000)
elif 'amount_proxy' in df.columns:
    pass
else:
    df['amount_proxy'] = 8000

df['is_large_trade'] = (df['amount_proxy'] >= 100000).astype(int)

print(f"    Monto promedio: ${df['amount_proxy'].mean():,.0f}")
print(f"    Trades grandes (>=100K): {df['is_large_trade'].mean()*100:.1f}%")

    Monto promedio: $33,291
    Trades grandes (>=100K): 5.1%


In [10]:
# === 2.3 TIMING ===
df['disclosure_delay'] = (df['filed_date'] - df['trade_date']).dt.days
df['disclosure_delay'] = df['disclosure_delay'].clip(lower=0, upper=365)
df['long_delay'] = (df['disclosure_delay'] > 30).astype(int)

df['day_of_month'] = df['trade_date'].dt.day
df['days_in_month'] = df['trade_date'].dt.daysinmonth
df['end_of_month'] = (df['days_in_month'] - df['day_of_month'] <= 5).astype(int)

df['day_of_week'] = df['trade_date'].dt.dayofweek
df['is_monday'] = (df['day_of_week'] == 0).astype(int)
df['is_friday'] = (df['day_of_week'] == 4).astype(int)

print(f"    Disclosure delay promedio: {df['disclosure_delay'].mean():.1f} días")
print(f"    Long delay (>30d): {df['long_delay'].mean()*100:.1f}%")

    Disclosure delay promedio: 43.7 días
    Long delay (>30d): 44.9%


In [11]:
# === 2.4 COMITÉS ===
def is_info_committee(committee_name):
    """Check if committee is informationally sensitive."""
    if pd.isna(committee_name):
        return 0
    for ic in INFO_COMMITTEES:
        if ic.lower() in str(committee_name).lower():
            return 1
    return 0

# Buscar columna de comité
COMM_COL = None
for col in ['committee_name', 'Committee', 'committee']:
    if col in df.columns:
        COMM_COL = col
        break

if COMM_COL:
    df['is_info_committee'] = df[COMM_COL].apply(is_info_committee)
    print(f"    Info committee trades: {df['is_info_committee'].mean()*100:.1f}%")
else:
    df['is_info_committee'] = 0
    print("    ⚠️ No se encontró columna de comité")

# Chair/Ranking member
if 'committee_role' in df.columns:
    df['is_chair'] = df['committee_role'].fillna('').str.lower().str.contains('chair|ranking', na=False).astype(int)
    print(f"    Chair/Ranking trades: {df['is_chair'].mean()*100:.1f}%")
else:
    df['is_chair'] = 0

    Info committee trades: 42.0%
    Chair/Ranking trades: 35.9%


In [12]:
# === 2.5 PODER DEL POLÍTICO ===

# Antigüedad
if 'Years in position' in df.columns:
    df['years_in_position'] = pd.to_numeric(df['Years in position'], errors='coerce').fillna(0)
elif 'seniority_years' in df.columns:
    df['years_in_position'] = pd.to_numeric(df['seniority_years'], errors='coerce').fillna(0)
else:
    df['years_in_position'] = 0

df['is_senior'] = (df['years_in_position'] >= 10).astype(int)

# Net worth
if 'Net worth' in df.columns:
    df['net_worth'] = pd.to_numeric(df['Net worth'], errors='coerce').fillna(0)
    median_nw = df.loc[df['net_worth'] > 0, 'net_worth'].median() if (df['net_worth'] > 0).any() else 0
    df['is_wealthy'] = (df['net_worth'] > median_nw).astype(int)
else:
    df['net_worth'] = 0
    df['is_wealthy'] = 0

# Senador
CHAMBER_COL = None
for col in ['chamber', 'Chamber']:
    if col in df.columns:
        CHAMBER_COL = col
        break

if CHAMBER_COL:
    df['is_senator'] = df[CHAMBER_COL].astype(str).str.lower().str.contains('senate').astype(int)
else:
    df['is_senator'] = 0

# Power index compuesto
df['power_index'] = df['is_chair'] + df['is_senator'] + df['is_senior'] + df['is_info_committee']

print(f"    Senadores: {df['is_senator'].mean()*100:.1f}%")
print(f"    Senior (>=10 años): {df['is_senior'].mean()*100:.1f}%")
print(f"    Power index promedio: {df['power_index'].mean():.2f}")

    Senadores: 0.0%
    Senior (>=10 años): 5.4%
    Power index promedio: 0.83


In [13]:
# === 2.6 PARTIDO ===
PARTY_COL = None
for col in ['party', 'Party', 'party_code']:
    if col in df.columns:
        PARTY_COL = col
        break

if PARTY_COL:
    df['is_democrat'] = df[PARTY_COL].astype(str).str.upper().str.contains('D|DEM').astype(int)
    df['is_republican'] = df[PARTY_COL].astype(str).str.upper().str.contains('R|REP').astype(int)
    print(f"    Demócratas: {df['is_democrat'].mean()*100:.1f}%")
    print(f"    Republicanos: {df['is_republican'].mean()*100:.1f}%")
else:
    df['is_democrat'] = 0
    df['is_republican'] = 0

    Demócratas: 49.0%
    Republicanos: 99.9%


In [14]:
# === 2.7 COMPORTAMIENTO ===

# Frequent trader
trader_counts = df.groupby(NAME_COL)['trade_date'].count()
frequent_threshold = trader_counts.quantile(0.75)
frequent_traders = set(trader_counts[trader_counts >= frequent_threshold].index)
df['frequent_trader'] = df[NAME_COL].isin(frequent_traders).astype(int)

# First time trading this stock
df = df.sort_values([NAME_COL, TICKER_COL, 'trade_date'])
df['first_time'] = (~df.duplicated(subset=[NAME_COL, TICKER_COL], keep='first')).astype(int)

# Direction change
df['prev_is_buy'] = df.groupby([NAME_COL, TICKER_COL])['is_buy'].shift(1)
df['direction_change'] = ((df['is_buy'] != df['prev_is_buy']) & df['prev_is_buy'].notna()).astype(int)

print(f"    Frequent traders: {df['frequent_trader'].mean()*100:.1f}%")
print(f"    First time trades: {df['first_time'].mean()*100:.1f}%")
print(f"    Direction changes: {df['direction_change'].mean()*100:.1f}%")

    Frequent traders: 93.5%
    First time trades: 17.9%
    Direction changes: 27.6%


In [15]:
# === 2.8 COORDINACIÓN ===
print("    Calculando coordinación...")

# Múltiples políticos misma acción mismo día
daily_traders = df.groupby(['trade_date', TICKER_COL])[NAME_COL].nunique().reset_index()
daily_traders.columns = ['trade_date', TICKER_COL, 'n_traders_same_day']
df = df.merge(daily_traders, on=['trade_date', TICKER_COL], how='left')
df['coordinated'] = (df['n_traders_same_day'] >= 2).astype(int)

# Coordinación por partido
if PARTY_COL:
    party_traders = df.groupby(['trade_date', TICKER_COL, PARTY_COL])[NAME_COL].nunique().reset_index()
    party_traders.columns = ['trade_date', TICKER_COL, PARTY_COL, 'n_party_traders']
    df = df.merge(party_traders, on=['trade_date', TICKER_COL, PARTY_COL], how='left')
    df['party_coordinated'] = (df['n_party_traders'] >= 2).astype(int)
else:
    df['party_coordinated'] = 0

# Coordinación por comité
if COMM_COL:
    # Solo considerar cuando hay comité no nulo
    df_with_comm = df[df[COMM_COL].notna()].copy()
    comm_traders = df_with_comm.groupby(['trade_date', TICKER_COL, COMM_COL])[NAME_COL].nunique().reset_index()
    comm_traders.columns = ['trade_date', TICKER_COL, COMM_COL, 'n_committee_traders']
    df = df.merge(comm_traders, on=['trade_date', TICKER_COL, COMM_COL], how='left')
    df['n_committee_traders'] = df['n_committee_traders'].fillna(1)
    df['committee_coordinated'] = (df['n_committee_traders'] >= 2).astype(int)
else:
    df['committee_coordinated'] = 0

print(f"    Trades coordinados (general): {df['coordinated'].mean()*100:.1f}%")
print(f"    Trades coordinados (partido): {df['party_coordinated'].mean()*100:.1f}%")
print(f"    Trades coordinados (comité): {df['committee_coordinated'].mean()*100:.1f}%")

    Calculando coordinación...
    Trades coordinados (general): 5.8%
    Trades coordinados (partido): 1.9%
    Trades coordinados (comité): 0.2%


In [16]:
# === 2.9 CONTEXTO DE MERCADO ===
print("    Calculando contexto de mercado...")

# Verificar columnas disponibles
has_momentum = 'momentum_20d' in df.columns
has_vol = 'realized_vol_30d' in df.columns
has_illiq = 'amihud_illiq_20d' in df.columns
has_mktcap = 'market_cap' in df.columns

print(f"      momentum_20d: {'✓' if has_momentum else '✗'}")
print(f"      realized_vol_30d: {'✓' if has_vol else '✗'}")
print(f"      amihud_illiq_20d: {'✓' if has_illiq else '✗'}")
print(f"      market_cap: {'✓' if has_mktcap else '✗'}")

# Contrarian: compra cuando cayó, vende cuando subió
if has_momentum:
    df['contrarian'] = (
        ((df['is_buy'] == 1) & (df['momentum_20d'] < 0)) |
        ((df['is_sell'] == 1) & (df['momentum_20d'] > 0))
    ).astype(int)
else:
    df['contrarian'] = 0

# High volatility
if has_vol:
    vol_median = df['realized_vol_30d'].median()
    df['high_vol'] = (df['realized_vol_30d'] > vol_median).astype(int)
else:
    df['high_vol'] = 0

# Illiquid
if has_illiq:
    illiq_median = df['amihud_illiq_20d'].median()
    df['illiquid'] = (df['amihud_illiq_20d'] > illiq_median).astype(int)
else:
    df['illiquid'] = 0

# Small cap
if has_mktcap:
    cap_median = df['market_cap'].median()
    df['small_cap'] = (df['market_cap'] < cap_median).astype(int)
else:
    df['small_cap'] = 0

if has_momentum:
    print(f"    Trades contrarian: {df['contrarian'].mean()*100:.1f}%")

    Calculando contexto de mercado...
      momentum_20d: ✓
      realized_vol_30d: ✓
      amihud_illiq_20d: ✓
      market_cap: ✓
    Trades contrarian: 38.6%


In [17]:
# === 2.10 SEÑALES COMPUESTAS ===
print("    Creando señales compuestas...")

# Smart money buy: compra + info committee + chair
df['smart_money_buy'] = (
    (df['is_buy'] == 1) & 
    (df['is_info_committee'] == 1) & 
    (df['is_chair'] == 1)
).astype(int)

# Smart money sell
df['smart_money_sell'] = (
    (df['is_sell'] == 1) & 
    (df['is_info_committee'] == 1) & 
    (df['is_chair'] == 1)
).astype(int)

# Insider ring: coordinación dentro del mismo comité informativo
df['insider_ring'] = (
    (df['committee_coordinated'] == 1) & 
    (df['is_info_committee'] == 1)
).astype(int)

# Hidden trade: ilíquido/small cap + info committee
df['hidden_trade'] = (
    ((df['illiquid'] == 1) | (df['small_cap'] == 1)) & 
    (df['is_info_committee'] == 1)
).astype(int)

# Opportunistic: contrarian + large + info committee
df['opportunistic'] = (
    (df['contrarian'] == 1) & 
    (df['is_large_trade'] == 1) & 
    (df['is_info_committee'] == 1)
).astype(int)

print(f"    Smart money buys: {df['smart_money_buy'].sum():,}")
print(f"    Insider ring: {df['insider_ring'].sum():,}")
print(f"    Hidden trades: {df['hidden_trade'].sum():,}")

    Creando señales compuestas...
    Smart money buys: 1,086
    Insider ring: 108
    Hidden trades: 16,630


---
## 3. Collapse: Congress Variables

In [18]:
print("\n[3] COLAPSANDO VARIABLES DE CONGRESO...")

# Políticos únicos por (ticker, month)
unique_politicians = df.groupby([TICKER_COL, 'trade_month'])[NAME_COL].nunique().reset_index()
unique_politicians.columns = ['ticker', 'month', 'cong_unique_politicians']

# Agregación principal
agg_cong = df.groupby([TICKER_COL, 'trade_month']).agg(
    # === CONTEOS BÁSICOS ===
    cong_total_trades=('is_buy', 'count'),
    cong_buy_count=('is_buy', 'sum'),
    cong_sell_count=('is_sell', 'sum'),
    
    # === MONTOS ===
    cong_total_amount=('amount_proxy', 'sum'),
    cong_avg_amount=('amount_proxy', 'mean'),
    cong_large_trades=('is_large_trade', 'sum'),
    
    # === TIMING ===
    cong_avg_disclosure_delay=('disclosure_delay', 'mean'),
    cong_max_disclosure_delay=('disclosure_delay', 'max'),
    cong_long_delay_trades=('long_delay', 'sum'),
    cong_end_of_month_trades=('end_of_month', 'sum'),
    cong_monday_trades=('is_monday', 'sum'),
    cong_friday_trades=('is_friday', 'sum'),
    
    # === COMITÉS ===
    cong_info_committee_trades=('is_info_committee', 'sum'),
    cong_chair_trades=('is_chair', 'sum'),
    
    # === PODER ===
    cong_senior_trades=('is_senior', 'sum'),
    cong_wealthy_trades=('is_wealthy', 'sum'),
    cong_senator_trades=('is_senator', 'sum'),
    cong_avg_power_index=('power_index', 'mean'),
    cong_max_power_index=('power_index', 'max'),
    cong_avg_seniority=('years_in_position', 'mean'),
    cong_avg_net_worth=('net_worth', 'mean'),
    
    # === PARTIDO ===
    cong_dem_trades=('is_democrat', 'sum'),
    cong_rep_trades=('is_republican', 'sum'),
    
    # === COMPORTAMIENTO ===
    cong_frequent_trader_trades=('frequent_trader', 'sum'),
    cong_first_time_trades=('first_time', 'sum'),
    cong_direction_change_trades=('direction_change', 'sum'),
    
    # === COORDINACIÓN ===
    cong_coordinated_trades=('coordinated', 'sum'),
    cong_party_coordinated_trades=('party_coordinated', 'sum'),
    cong_committee_coordinated_trades=('committee_coordinated', 'sum'),
    cong_max_traders_same_day=('n_traders_same_day', 'max'),
    
    # === CONTEXTO DE MERCADO ===
    cong_contrarian_trades=('contrarian', 'sum'),
    cong_high_vol_trades=('high_vol', 'sum'),
    cong_illiquid_trades=('illiquid', 'sum'),
    cong_small_cap_trades=('small_cap', 'sum'),
    
    # === SEÑALES COMPUESTAS ===
    cong_smart_money_buy_trades=('smart_money_buy', 'sum'),
    cong_smart_money_sell_trades=('smart_money_sell', 'sum'),
    cong_insider_ring_trades=('insider_ring', 'sum'),
    cong_hidden_trades=('hidden_trade', 'sum'),
    cong_opportunistic_trades=('opportunistic', 'sum'),
    
).reset_index()

agg_cong.columns = ['ticker', 'month'] + list(agg_cong.columns[2:])

# Merge unique politicians
agg_cong = agg_cong.merge(unique_politicians, on=['ticker', 'month'], how='left')

print(f"    Observaciones: {len(agg_cong):,}")


[3] COLAPSANDO VARIABLES DE CONGRESO...
    Observaciones: 43,381


In [19]:
# === VARIABLES DERIVADAS ===
print("    Creando ratios y variables derivadas...")

total = agg_cong['cong_total_trades']

# Señal neta
agg_cong['cong_net'] = agg_cong['cong_buy_count'] - agg_cong['cong_sell_count']
agg_cong['cong_buy_ratio'] = agg_cong['cong_buy_count'] / total
agg_cong['cong_csi'] = agg_cong['cong_net'] / total  # Congressional Sentiment Index [-1, 1]

# Ratios sobre total
agg_cong['cong_info_ratio'] = agg_cong['cong_info_committee_trades'] / total
agg_cong['cong_chair_ratio'] = agg_cong['cong_chair_trades'] / total
agg_cong['cong_senior_ratio'] = agg_cong['cong_senior_trades'] / total
agg_cong['cong_senator_ratio'] = agg_cong['cong_senator_trades'] / total
agg_cong['cong_dem_ratio'] = agg_cong['cong_dem_trades'] / total
agg_cong['cong_rep_ratio'] = agg_cong['cong_rep_trades'] / total
agg_cong['cong_coordinated_ratio'] = agg_cong['cong_coordinated_trades'] / total
agg_cong['cong_party_coordinated_ratio'] = agg_cong['cong_party_coordinated_trades'] / total
agg_cong['cong_committee_coordinated_ratio'] = agg_cong['cong_committee_coordinated_trades'] / total
agg_cong['cong_first_time_ratio'] = agg_cong['cong_first_time_trades'] / total
agg_cong['cong_large_ratio'] = agg_cong['cong_large_trades'] / total
agg_cong['cong_long_delay_ratio'] = agg_cong['cong_long_delay_trades'] / total
agg_cong['cong_frequent_trader_ratio'] = agg_cong['cong_frequent_trader_trades'] / total

# Ratios de contexto
agg_cong['cong_contrarian_ratio'] = agg_cong['cong_contrarian_trades'] / total
agg_cong['cong_high_vol_ratio'] = agg_cong['cong_high_vol_trades'] / total
agg_cong['cong_illiquid_ratio'] = agg_cong['cong_illiquid_trades'] / total
agg_cong['cong_small_cap_ratio'] = agg_cong['cong_small_cap_trades'] / total

# Intensidad
agg_cong['cong_intensity'] = total / agg_cong['cong_unique_politicians']

# Señales binarias
agg_cong['cong_consensus_buy'] = (agg_cong['cong_buy_ratio'] > 0.7).astype(int)
agg_cong['cong_consensus_sell'] = (agg_cong['cong_buy_ratio'] < 0.3).astype(int)
agg_cong['cong_multiple_politicians'] = (agg_cong['cong_unique_politicians'] > 1).astype(int)
agg_cong['cong_bipartisan'] = ((agg_cong['cong_dem_trades'] > 0) & (agg_cong['cong_rep_trades'] > 0)).astype(int)

# Señales compuestas agregadas
agg_cong['cong_smart_money'] = ((agg_cong['cong_net'] > 0) & 
                                (agg_cong['cong_info_committee_trades'] > 0) & 
                                (agg_cong['cong_chair_trades'] > 0)).astype(int)

agg_cong['cong_strong_buy'] = ((agg_cong['cong_csi'] > 0.5) & 
                               (agg_cong['cong_unique_politicians'] >= 2)).astype(int)

agg_cong['cong_strong_sell'] = ((agg_cong['cong_csi'] < -0.5) & 
                                (agg_cong['cong_unique_politicians'] >= 2)).astype(int)

# Señales binarias de compuestas
agg_cong['cong_has_insider_ring'] = (agg_cong['cong_insider_ring_trades'] > 0).astype(int)
agg_cong['cong_has_hidden'] = (agg_cong['cong_hidden_trades'] > 0).astype(int)
agg_cong['cong_has_opportunistic'] = (agg_cong['cong_opportunistic_trades'] > 0).astype(int)

print(f"    Variables de congreso: {len([c for c in agg_cong.columns if c.startswith('cong_')])}")

    Creando ratios y variables derivadas...
    Variables de congreso: 71


---
## 4. Collapse: Market Variables (End of Month)

In [20]:
print("\n[4] COLAPSANDO VARIABLES DE MERCADO...")

# Identificar variables de mercado disponibles
MKT_VARS_CANDIDATES = [
    # Returns
    'return_t', 'excess_return',
    # Momentum
    'momentum_5d', 'momentum_20d', 'momentum_60d', 'momentum_252d',
    # Volatility
    'realized_vol_30d', 'realized_vol_60d', 'realized_vol_252d',
    'parkinson_vol_30d', 'vol_of_vol_60d',
    # Liquidity
    'volume_ratio_30d', 'abnormal_volume_30d',
    'amihud_illiq_20d', 'roll_spread_30d', 'hl_spread_20d', 'zero_volume_days_30d',
    # Factor exposures
    'beta_252d', 'r2_market_252d',
    'alpha_ff3_252d', 'beta_mkt_ff3_252d', 'beta_smb_ff3_252d', 'beta_hml_ff3_252d', 'r2_ff3_252d',
    # Fundamentals
    'market_cap', 'price', 'book_value', 'price_to_book', 'ev_to_ebitda',
    # CAR (post-trade returns - these ARE the outcome at trade level)
    'car_raw_30d', 'car_capm_30d', 'car_ff3_30d',
    'car_raw_60d', 'car_capm_60d', 'car_ff3_60d',
    'car_raw_90d', 'car_capm_90d', 'car_ff3_90d',
]

MKT_VARS = [v for v in MKT_VARS_CANDIDATES if v in df.columns]
print(f"    Variables de mercado disponibles: {len(MKT_VARS)}/{len(MKT_VARS_CANDIDATES)}")


[4] COLAPSANDO VARIABLES DE MERCADO...
    Variables de mercado disponibles: 38/38


In [21]:
# Estrategia: usar el ÚLTIMO trade del mes como proxy del fin de mes
# Esto evita el problema de promediar diferentes puntos en el tiempo

print("    Usando último trade del mes como proxy de fin de mes...")

# Ordenar y tomar el último trade de cada (ticker, month)
df_sorted = df.sort_values(['trade_date'])
last_trade_idx = df_sorted.groupby([TICKER_COL, 'trade_month'])['trade_date'].idxmax()
df_last = df_sorted.loc[last_trade_idx]

# Seleccionar columnas de mercado
cols_to_keep = [TICKER_COL, 'trade_month'] + MKT_VARS
cols_available = [c for c in cols_to_keep if c in df_last.columns]
agg_mkt = df_last[cols_available].copy()
agg_mkt.columns = ['ticker', 'month'] + ['mkt_' + c for c in cols_available[2:]]

print(f"    Observaciones: {len(agg_mkt):,}")

    Usando último trade del mes como proxy de fin de mes...
    Observaciones: 43,381


In [22]:
# También calcular promedio ponderado por monto de CARs (para variable objetivo alternativa)
print("    Calculando CAR promedio ponderado por monto...")

car_vars = ['car_raw_30d', 'car_capm_30d', 'car_ff3_30d']
car_available = [c for c in car_vars if c in df.columns]

if car_available:
    def weighted_mean(group, value_col, weight_col='amount_proxy'):
        """Weighted mean, fallback to simple mean if weights are zero."""
        d = group[[value_col, weight_col]].dropna()
        if len(d) == 0 or d[weight_col].sum() == 0:
            return np.nan
        return np.average(d[value_col], weights=d[weight_col])
    
    car_weighted = []
    for (ticker, month), group in df.groupby([TICKER_COL, 'trade_month']):
        row = {'ticker': ticker, 'month': month}
        for car_var in car_available:
            row[f'mkt_{car_var}_wavg'] = weighted_mean(group, car_var)
        car_weighted.append(row)
    
    df_car_wavg = pd.DataFrame(car_weighted)
    agg_mkt = agg_mkt.merge(df_car_wavg, on=['ticker', 'month'], how='left')
    print(f"    CARs ponderados añadidos: {len(car_available)}")

    Calculando CAR promedio ponderado por monto...
    CARs ponderados añadidos: 3


---
## 5. Merge & Create Panel

In [23]:
print("\n[5] MERGEANDO CONGRESO + MERCADO...")

panel = agg_cong.merge(agg_mkt, on=['ticker', 'month'], how='left')

print(f"    Observaciones: {len(panel):,}")
print(f"    Variables: {len(panel.columns)}")


[5] MERGEANDO CONGRESO + MERCADO...
    Observaciones: 43,381
    Variables: 114


---
## 6. Target Variables

In [24]:
print("\n[6] CREANDO VARIABLES OBJETIVO...")

# Ordenar por ticker y mes
panel = panel.sort_values(['ticker', 'month'])

# === OPCIÓN 1: Retorno del mes siguiente ===
# Usar mkt_return_t shifteado (retorno del mes t como proxy del retorno del mes t+1)
if 'mkt_return_t' in panel.columns:
    panel['ret_future_1m'] = panel.groupby('ticker')['mkt_return_t'].shift(-1)
    print(f"    ret_future_1m: {panel['ret_future_1m'].notna().sum():,} obs con valor")
else:
    print("    ⚠️ mkt_return_t no disponible")

# === OPCIÓN 2: CAR como variable objetivo ===
# El CAR ya es post-trade, así que NO necesita shift
# Usamos el CAR del último trade del mes como proxy
if 'mkt_car_ff3_30d' in panel.columns:
    panel['ret_future_car30_ff3'] = panel['mkt_car_ff3_30d']
    print(f"    ret_future_car30_ff3: {panel['ret_future_car30_ff3'].notna().sum():,} obs con valor")

if 'mkt_car_raw_30d' in panel.columns:
    panel['ret_future_car30_raw'] = panel['mkt_car_raw_30d']
    print(f"    ret_future_car30_raw: {panel['ret_future_car30_raw'].notna().sum():,} obs con valor")

# === OPCIÓN 3: CAR promedio ponderado ===
if 'mkt_car_ff3_30d_wavg' in panel.columns:
    panel['ret_future_car30_ff3_wavg'] = panel['mkt_car_ff3_30d_wavg']
    print(f"    ret_future_car30_ff3_wavg: {panel['ret_future_car30_ff3_wavg'].notna().sum():,} obs con valor")


[6] CREANDO VARIABLES OBJETIVO...
    ret_future_1m: 32,169 obs con valor
    ret_future_car30_ff3: 34,623 obs con valor
    ret_future_car30_raw: 34,748 obs con valor
    ret_future_car30_ff3_wavg: 34,624 obs con valor


In [25]:
# Crear month_dt para ordenamiento temporal
panel['month_dt'] = pd.to_datetime(panel['month'].astype(str))

print(f"\n    Variables objetivo disponibles:")
target_vars = [c for c in panel.columns if c.startswith('ret_future')]
for tv in target_vars:
    print(f"      - {tv}")


    Variables objetivo disponibles:
      - ret_future_1m
      - ret_future_car30_ff3
      - ret_future_car30_raw
      - ret_future_car30_ff3_wavg


---
## 7. Winsorizing & Validation

In [26]:
print("\n[7] WINSORIZING Y VALIDACIÓN...")

# Identificar columnas numéricas (excluyendo identificadores)
id_cols = ['ticker', 'month', 'month_dt']
numeric_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
cols_to_winsorize = [c for c in numeric_cols if c not in id_cols]

print(f"    Columnas a winsorizar: {len(cols_to_winsorize)}")


[7] WINSORIZING Y VALIDACIÓN...
    Columnas a winsorizar: 116


In [27]:
# Winsorizing
def winsorize_column(series, pctl=0.01):
    """Winsorize a series at pctl and 1-pctl."""
    if series.isna().all():
        return series
    lower = series.quantile(pctl)
    upper = series.quantile(1 - pctl)
    return series.clip(lower=lower, upper=upper)

print(f"    Winsorizing at {WINSOR_PCTL*100:.0f}% / {(1-WINSOR_PCTL)*100:.0f}%...")

winsor_report = []
for col in cols_to_winsorize:
    original_min = panel[col].min()
    original_max = panel[col].max()
    
    panel[col] = winsorize_column(panel[col], WINSOR_PCTL)
    
    new_min = panel[col].min()
    new_max = panel[col].max()
    
    if original_min != new_min or original_max != new_max:
        winsor_report.append({
            'column': col,
            'original_min': original_min,
            'original_max': original_max,
            'new_min': new_min,
            'new_max': new_max
        })

print(f"    Columnas modificadas: {len(winsor_report)}")

    Winsorizing at 1% / 99%...
    Columnas modificadas: 90


In [28]:
# Mostrar ejemplos de winsorizing
if winsor_report:
    print("\n    Ejemplos de winsorizing:")
    for row in winsor_report[:10]:
        print(f"      {row['column']}: [{row['original_min']:.4f}, {row['original_max']:.4f}] → [{row['new_min']:.4f}, {row['new_max']:.4f}]")


    Ejemplos de winsorizing:
      cong_total_trades: [1.0000, 60.0000] → [1.0000, 11.0000]
      cong_buy_count: [0.0000, 60.0000] → [0.0000, 7.0000]
      cong_sell_count: [0.0000, 48.0000] → [0.0000, 7.0000]
      cong_total_amount: [8000.0000, 33215000.0000] → [8000.0000, 736700.0000]
      cong_avg_amount: [8000.0000, 3000000.0000] → [8000.0000, 275000.0000]
      cong_large_trades: [0.0000, 27.0000] → [0.0000, 2.0000]
      cong_avg_disclosure_delay: [0.0000, 365.0000] → [3.0000, 365.0000]
      cong_max_disclosure_delay: [0.0000, 365.0000] → [3.0000, 365.0000]
      cong_long_delay_trades: [0.0000, 60.0000] → [0.0000, 7.0000]
      cong_end_of_month_trades: [0.0000, 27.0000] → [0.0000, 4.0000]


In [29]:
# Validación de NaN
print("\n    Validando NaN...")

nan_report = panel.isnull().sum()
nan_cols = nan_report[nan_report > 0].sort_values(ascending=False)

if len(nan_cols) > 0:
    print(f"    Columnas con NaN: {len(nan_cols)}")
    print("\n    Top 15 columnas con más NaN:")
    for col, count in nan_cols.head(15).items():
        pct = count / len(panel) * 100
        print(f"      {col}: {count:,} ({pct:.1f}%)")
else:
    print("    ✓ No hay NaN")


    Validando NaN...
    Columnas con NaN: 45

    Top 15 columnas con más NaN:
      mkt_ev_to_ebitda: 13,549 (31.2%)
      ret_future_1m: 11,212 (25.8%)
      mkt_price_to_book: 9,833 (22.7%)
      mkt_book_value: 9,788 (22.6%)
      mkt_market_cap: 9,739 (22.4%)
      mkt_price: 9,351 (21.6%)
      mkt_momentum_252d: 9,028 (20.8%)
      mkt_realized_vol_252d: 9,025 (20.8%)
      mkt_amihud_illiq_20d: 8,832 (20.4%)
      mkt_abnormal_volume_30d: 8,785 (20.3%)
      mkt_volume_ratio_30d: 8,785 (20.3%)
      mkt_car_ff3_90d: 8,769 (20.2%)
      mkt_car_capm_90d: 8,769 (20.2%)
      mkt_car_ff3_60d: 8,762 (20.2%)
      mkt_car_capm_60d: 8,762 (20.2%)


In [30]:
# Rellenar NaN en ratios con 0 (división por cero)
ratio_cols = [c for c in panel.columns if 'ratio' in c.lower()]
for col in ratio_cols:
    panel[col] = panel[col].fillna(0)

# Verificar infinitos
print("\n    Verificando infinitos...")
numeric_df = panel.select_dtypes(include=[np.number])
inf_counts = np.isinf(numeric_df).sum()
inf_cols = inf_counts[inf_counts > 0]

if len(inf_cols) > 0:
    print(f"    Columnas con infinitos: {len(inf_cols)}")
    for col, count in inf_cols.items():
        panel[col] = panel[col].replace([np.inf, -np.inf], np.nan)
    print("    Infinitos reemplazados con NaN")
else:
    print("    ✓ No hay infinitos")


    Verificando infinitos...
    ✓ No hay infinitos


In [31]:
# Validación de rangos esperados
print("\n    Validando rangos esperados...")

validations = [
    ('cong_csi', -1, 1, 'CSI debe estar en [-1, 1]'),
    ('cong_buy_ratio', 0, 1, 'Buy ratio debe estar en [0, 1]'),
    ('cong_info_ratio', 0, 1, 'Info ratio debe estar en [0, 1]'),
    ('cong_total_trades', 1, None, 'Total trades debe ser >= 1'),
    ('cong_unique_politicians', 1, None, 'Unique politicians debe ser >= 1'),
]

for col, min_val, max_val, msg in validations:
    if col not in panel.columns:
        continue
    
    violations = 0
    if min_val is not None:
        violations += (panel[col] < min_val).sum()
    if max_val is not None:
        violations += (panel[col] > max_val).sum()
    
    if violations > 0:
        print(f"    ⚠️ {col}: {violations:,} violaciones - {msg}")
    else:
        print(f"    ✓ {col}: OK")


    Validando rangos esperados...
    ✓ cong_csi: OK
    ✓ cong_buy_ratio: OK
    ✓ cong_info_ratio: OK
    ✓ cong_total_trades: OK
    ✓ cong_unique_politicians: OK


---
## 8. Export

In [32]:
print("\n[8] EXPORTANDO...")

# Definir feature sets
FEATURES_CONGRESS = [c for c in panel.columns if c.startswith('cong_')]
FEATURES_MARKET = [c for c in panel.columns if c.startswith('mkt_') and 'car_' not in c]
FEATURES_ALL = FEATURES_CONGRESS + FEATURES_MARKET
TARGET_VARS = [c for c in panel.columns if c.startswith('ret_future')]

print(f"    Features de congreso: {len(FEATURES_CONGRESS)}")
print(f"    Features de mercado: {len(FEATURES_MARKET)}")
print(f"    Variables objetivo: {len(TARGET_VARS)}")


[8] EXPORTANDO...
    Features de congreso: 71
    Features de mercado: 29
    Variables objetivo: 4


In [33]:
# Guardar panel completo
output_path = os.path.join(OUTPUT_DIR, 'panel_final_stock_month.parquet')
panel.to_parquet(output_path, index=False)
print(f"    Guardado: {output_path}")

# CSV para inspección
csv_path = os.path.join(OUTPUT_DIR, 'panel_final_stock_month.csv')
panel.to_csv(csv_path, index=False)
print(f"    Guardado: {csv_path}")

# Feature sets
feature_sets = {
    'congress': FEATURES_CONGRESS,
    'market': FEATURES_MARKET,
    'all': FEATURES_ALL,
    'targets': TARGET_VARS
}
features_path = os.path.join(OUTPUT_DIR, 'feature_sets.json')
with open(features_path, 'w') as f:
    json.dump(feature_sets, f, indent=2)
print(f"    Guardado: {features_path}")

# Winsorizing report
if winsor_report:
    winsor_df = pd.DataFrame(winsor_report)
    winsor_path = os.path.join(OUTPUT_DIR, 'winsorizing_report.csv')
    winsor_df.to_csv(winsor_path, index=False)
    print(f"    Guardado: {winsor_path}")

    Guardado: data/prediction_bases/panel_final_stock_month.parquet
    Guardado: data/prediction_bases/panel_final_stock_month.csv
    Guardado: data/prediction_bases/feature_sets.json
    Guardado: data/prediction_bases/winsorizing_report.csv


---
## 9. Summary

In [34]:
print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)

print(f"""
DIMENSIONES:
  Observaciones:     {len(panel):,}
  Acciones únicas:   {panel['ticker'].nunique():,}
  Meses:             {panel['month'].nunique()}
  Período:           {panel['month'].min()} a {panel['month'].max()}

VARIABLES:
  Totales:           {len(panel.columns)}
  Congreso (cong_):  {len(FEATURES_CONGRESS)}
  Mercado (mkt_):    {len(FEATURES_MARKET)}
  Target (ret_):     {len(TARGET_VARS)}

VARIABLES OBJETIVO DISPONIBLES:
  ret_future_1m:          Retorno del mes siguiente
  ret_future_car30_ff3:   CAR FF3 30d (último trade del mes)
  ret_future_car30_raw:   CAR raw 30d (último trade del mes)
  ret_future_car30_ff3_wavg: CAR FF3 30d (promedio ponderado por monto)

PROCESAMIENTO:
  Winsorizing:       {WINSOR_PCTL*100:.0f}% / {(1-WINSOR_PCTL)*100:.0f}%
  NaN en ratios:     Rellenados con 0
  Infinitos:         Reemplazados con NaN

ARCHIVOS GENERADOS:
  - panel_final_stock_month.parquet
  - panel_final_stock_month.csv
  - feature_sets.json
  - winsorizing_report.csv
""")

print("="*70)
print("✅ PANEL LISTO PARA MODELADO")
print("="*70)


RESUMEN FINAL

DIMENSIONES:
  Observaciones:     43,381
  Acciones únicas:   4,440
  Meses:             150
  Período:           2012-06 a 2024-12

VARIABLES:
  Totales:           119
  Congreso (cong_):  71
  Mercado (mkt_):    29
  Target (ret_):     4

VARIABLES OBJETIVO DISPONIBLES:
  ret_future_1m:          Retorno del mes siguiente
  ret_future_car30_ff3:   CAR FF3 30d (último trade del mes)
  ret_future_car30_raw:   CAR raw 30d (último trade del mes)
  ret_future_car30_ff3_wavg: CAR FF3 30d (promedio ponderado por monto)

PROCESAMIENTO:
  Winsorizing:       1% / 99%
  NaN en ratios:     Rellenados con 0
  Infinitos:         Reemplazados con NaN

ARCHIVOS GENERADOS:
  - panel_final_stock_month.parquet
  - panel_final_stock_month.csv
  - feature_sets.json
  - winsorizing_report.csv

✅ PANEL LISTO PARA MODELADO


In [35]:
# Estadísticas descriptivas de variables clave
print("\n📊 ESTADÍSTICAS DE VARIABLES CLAVE:")

print("\n--- Variables Objetivo ---")
if TARGET_VARS:
    print(panel[TARGET_VARS].describe().round(4))

print("\n--- Features de Congreso (selección) ---")
cong_key = ['cong_total_trades', 'cong_net', 'cong_csi', 'cong_unique_politicians', 
            'cong_info_ratio', 'cong_avg_power_index', 'cong_coordinated_ratio']
cong_key = [c for c in cong_key if c in panel.columns]
print(panel[cong_key].describe().round(3))

print("\n--- Features de Mercado (selección) ---")
mkt_key = ['mkt_momentum_20d', 'mkt_realized_vol_30d', 'mkt_market_cap', 'mkt_beta_252d']
mkt_key = [c for c in mkt_key if c in panel.columns]
if mkt_key:
    print(panel[mkt_key].describe().round(3))


📊 ESTADÍSTICAS DE VARIABLES CLAVE:

--- Variables Objetivo ---
       ret_future_1m  ret_future_car30_ff3  ret_future_car30_raw  \
count     32169.0000            34623.0000            34748.0000   
mean          0.0006               -0.0013                0.0008   
std           0.0217                0.0914                0.0965   
min          -0.0748               -0.2746               -0.2752   
25%          -0.0094               -0.0500               -0.0527   
50%           0.0007               -0.0004               -0.0003   
75%           0.0111                0.0475                0.0518   
max           0.0706                0.2908                0.3162   

       ret_future_car30_ff3_wavg  
count                 34624.0000  
mean                     -0.0018  
std                       0.0896  
min                      -0.2710  
25%                      -0.0493  
50%                      -0.0008  
75%                       0.0466  
max                       0.2834  

--- Fea

In [36]:
# Correlaciones con variable objetivo principal
print("\n--- Top 15 Correlaciones con ret_future_car30_ff3 ---")

if 'ret_future_car30_ff3' in panel.columns:
    numeric_cols = panel.select_dtypes(include=[np.number]).columns
    numeric_cols = [c for c in numeric_cols if c != 'ret_future_car30_ff3']
    
    correlations = {}
    for col in numeric_cols:
        valid = panel[[col, 'ret_future_car30_ff3']].dropna()
        if len(valid) > 100:
            correlations[col] = valid[col].corr(valid['ret_future_car30_ff3'])
    
    corr_sorted = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    for col, corr in corr_sorted[:15]:
        print(f"  {col}: {corr:.4f}")


--- Top 15 Correlaciones con ret_future_car30_ff3 ---
  cong_info_committee_trades: 0.0147
  cong_wealthy_trades: nan
  cong_senator_trades: nan
  cong_avg_net_worth: nan
  cong_first_time_trades: -0.0168
  cong_coordinated_trades: 0.0136
  cong_committee_coordinated_trades: nan
  cong_small_cap_trades: -0.0239
  cong_avg_power_index: 0.0141
  cong_max_traders_same_day: 0.0136
  cong_insider_ring_trades: nan
  cong_opportunistic_trades: nan
  cong_info_ratio: 0.0156
  cong_max_power_index: 0.0140
  cong_avg_seniority: 0.0120


In [37]:
# Listar todas las columnas
print("\n--- TODAS LAS COLUMNAS ---")
for i, col in enumerate(sorted(panel.columns)):
    print(f"  {i+1:2}. {col}")


--- TODAS LAS COLUMNAS ---
   1. cong_avg_amount
   2. cong_avg_disclosure_delay
   3. cong_avg_net_worth
   4. cong_avg_power_index
   5. cong_avg_seniority
   6. cong_bipartisan
   7. cong_buy_count
   8. cong_buy_ratio
   9. cong_chair_ratio
  10. cong_chair_trades
  11. cong_committee_coordinated_ratio
  12. cong_committee_coordinated_trades
  13. cong_consensus_buy
  14. cong_consensus_sell
  15. cong_contrarian_ratio
  16. cong_contrarian_trades
  17. cong_coordinated_ratio
  18. cong_coordinated_trades
  19. cong_csi
  20. cong_dem_ratio
  21. cong_dem_trades
  22. cong_direction_change_trades
  23. cong_end_of_month_trades
  24. cong_first_time_ratio
  25. cong_first_time_trades
  26. cong_frequent_trader_ratio
  27. cong_frequent_trader_trades
  28. cong_friday_trades
  29. cong_has_hidden
  30. cong_has_insider_ring
  31. cong_has_opportunistic
  32. cong_hidden_trades
  33. cong_high_vol_ratio
  34. cong_high_vol_trades
  35. cong_illiquid_ratio
  36. cong_illiquid_trades
 